<a href="https://colab.research.google.com/github/sameekshya1999/qpe-spectral-resolvability/blob/main/qpe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
!pip install -q qiskit qiskit-aer scipy pandas scikit-learn

import os, time, warnings
import numpy as np
import pandas as pd
from scipy.linalg import schur
from scipy.stats import unitary_group
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
warnings.filterwarnings("ignore")

# ---------------- Settings ----------------
b = 2                        # target qubits -> 4x4 gates
d = 2**b                     # 4 eigenvalues per gate
A_SWEEP = [3, 4, 5]          # ancillas -> 8, 16, 32 slices (main = 4)
SEEDS = [42, 43, 44]         # independent repeats (new gates + new shots each time)
SHOTS = 4096
N_HAAR, N_OTHER = 1000, 250
PEAK_THR = 0.05              # main peak threshold
THR_CHECK = [0.03, 0.08]     # extra thresholds for sensitivity check
USE_NOISE = False            # main run is ideal; True = small noise check
P1Q, P2Q, P_CU, P_READ = 0.001, 0.01, 0.03, 0.02
CHUNK = 250

# ---------------- Where to save ----------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks"   # change if your .ipynb is elsewhere
except ImportError:
    SAVE_DIR = os.getcwd()
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------- Gates ----------------
STANDARD = {
    "CNOT":     np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex),
    "SWAP":     np.array([[1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=complex),
    "iSWAP":    np.array([[1,0,0,0],[0,0,1j,0],[0,1j,0,0],[0,0,0,1]], dtype=complex),
    "sqrtSWAP": np.array([[1,0,0,0],[0,(1+1j)/2,(1-1j)/2,0],
                          [0,(1-1j)/2,(1+1j)/2,0],[0,0,0,1]], dtype=complex),
    "CZ":       np.diag([1,1,1,-1]).astype(complex),
    "QFT":      0.5*np.array([[1,1,1,1],[1,1j,-1,-1j],[1,-1,1,-1],[1,-1j,-1,1j]], dtype=complex),
}

def make_gates(rng):
    def new_seed():
        return int(rng.integers(0, 2**31 - 1))
    def from_phases(ph):
        V = unitary_group.rvs(d, random_state=new_seed())
        return V @ np.diag(np.exp(1j * ph)) @ V.conj().T
    def haar():
        return unitary_group.rvs(d, random_state=new_seed())
    def uniform():               # evenly spaced + 0.1 rad jitter
        ph = rng.uniform(0, 2*np.pi) + 2*np.pi*np.arange(d)/d + rng.normal(0, 0.10, d)
        return from_phases(np.mod(ph, 2*np.pi))
    def clustered():             # two opposite centers, random offset
        c = rng.uniform(0, 2*np.pi)
        return from_phases(np.mod(rng.choice([c, c + np.pi], size=d) + rng.normal(0, 0.15, d), 2*np.pi))
    def large_gap():             # every gap >= half the uniform gap
        mg = 0.5 * 2*np.pi / d
        gaps = mg + (2*np.pi - d*mg) * rng.dirichlet(np.ones(d))
        return from_phases(np.mod(rng.uniform(0, 2*np.pi) + np.cumsum(gaps), 2*np.pi))
    g = [(k, "standard", U) for k, U in STANDARD.items()]
    g += [(f"haar_{i}", "haar", haar()) for i in range(N_HAAR)]
    for cls, mk in [("uniform", uniform), ("clustered", clustered), ("large_gap", large_gap)]:
        g += [(f"{cls}_{i}", cls, mk()) for i in range(N_OTHER)]
    return g

def eigen_decomp(U):
    T, Z = schur(U, output="complex")             # orthonormal eigenvectors, even if degenerate
    return np.mod(np.angle(np.diag(T)), 2*np.pi), Z

# ---------------- Features (X) ----------------
MAX_GAP_VAR = (2*np.pi)**2 * (d - 1) / d**2

def features(phases, S):
    ph = np.sort(phases)
    gaps = np.diff(np.append(ph, ph[0] + 2*np.pi))
    R = np.abs(np.mean(np.exp(1j * ph)))
    u = phases * S / (2*np.pi)
    delta = np.abs(u - np.round(u))
    g = np.sort(gaps)
    return {"SCM": 1 - np.var(gaps) / MAX_GAP_VAR, "gap": g[0],
            "phase_std": np.sqrt(2 * (1 - R)),
            "grid_mean": delta.mean(), "grid_max": delta.max(),
            "g1": g[0], "g2": g[1], "g3": g[2], "g4": g[3]}   # all circular gaps, sorted (for baseline)

# ---------------- Targets (y) ----------------
def circ_dist(x, y):
    diff = abs(x - y) % (2*np.pi)
    return min(diff, 2*np.pi - diff)

def peaks(P, S, thr):
    return [k for k in range(S) if P[k] >= thr and P[k] > P[(k-1) % S] and P[k] >= P[(k+1) % S]]

def ideal_probs(phases, S):
    x, xp = np.arange(S), np.arange(S)[:, None]
    return sum(0.25 * np.abs(np.exp(1j*x*(th - 2*np.pi*xp/S)).sum(axis=1) / S)**2 for th in phases)

# ---------------- QPE circuit ----------------
P0, P1, I_d = np.diag([1, 0]), np.diag([0, 1]), np.eye(d)

def prep_unitary(v, r):
    A = np.column_stack([v, r.normal(size=(d, d-1)) + 1j*r.normal(size=(d, d-1))])
    Q, Rm = np.linalg.qr(A)
    Q[:, 0] *= Rm[0, 0]
    return Q

def inverse_qft(qc, q):
    m = len(q)
    for i in range(m // 2):
        qc.swap(q[i], q[m-1-i])
    for j in range(m):
        for k in range(j):
            qc.cp(-np.pi / 2**(j-k), q[k], q[j])
        qc.h(q[j])

def qpe_circuit(phases, Z, a, r):
    anc, tgt = QuantumRegister(a, "anc"), QuantumRegister(b, "tgt")
    c = ClassicalRegister(a, "c")
    qc = QuantumCircuit(anc, tgt, c)
    v = Z @ (np.ones(d) / np.sqrt(d))                         # equal mix of 4 eigenvectors
    qc.append(UnitaryGate(prep_unitary(v, r), label="prep", check_input=False), tgt)
    qc.h(anc)
    for j in range(a):
        Uj = Z @ np.diag(np.exp(1j * (2**j) * phases)) @ Z.conj().T
        qc.append(UnitaryGate(np.kron(Uj, P1) + np.kron(I_d, P0), label="cu", check_input=False),
                  [anc[j], *tgt])
    inverse_qft(qc, list(anc))
    qc.measure(anc, c)
    return qc

if USE_NOISE:
    nm = NoiseModel(); nm.add_basis_gates(["unitary"])
    nm.add_all_qubit_quantum_error(depolarizing_error(P1Q, 1), ["h"])
    nm.add_all_qubit_quantum_error(depolarizing_error(P2Q, 2), ["cp"])
    nm.add_all_qubit_quantum_error(depolarizing_error(3*P2Q, 2), ["swap"])
    nm.add_all_qubit_quantum_error(depolarizing_error(P_CU, 3), ["cu"])
    nm.add_all_qubit_readout_error(ReadoutError([[1-P_READ, P_READ], [P_READ, 1-P_READ]]))
    sim = AerSimulator(method="density_matrix", noise_model=nm)
else:
    sim = AerSimulator()

# ---------------- Run ----------------
rows, t_all = [], time.time()
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    gates = make_gates(rng)
    dec = [eigen_decomp(U) for _, _, U in gates]
    for a in A_SWEEP:
        S, prep_rng, t0 = 2**a, np.random.default_rng(seed + 1), time.time()
        for start in range(0, len(gates), CHUNK):
            idx = range(start, min(start + CHUNK, len(gates)))
            circs = [qpe_circuit(dec[g][0], dec[g][1], a, prep_rng) for g in idx]
            res = sim.run(circs, shots=SHOTS, seed_simulator=seed * 1000 + a * 100 + start).result()
            for i, g in enumerate(idx):
                name, cls, _ = gates[g]
                phases = dec[g][0]
                P = np.zeros(S)
                for key, val in res.get_counts(i).items():
                    P[int(key, 2)] = val / SHOTS
                pk = peaks(P, S, PEAK_THR)
                n_pk = min(len(pk), d)
                err = (np.mean([min(circ_dist(2*np.pi*k/S, th) for k in pk) for th in phases])
                       if pk else np.pi)
                row = {"seed": seed, "a": a, "slices": S, "gate": name, "class": cls,
                       **features(phases, S),
                       "y_peaks": n_pk, "y_resolved": n_pk / d, "y_phase_error": err,
                       "y_resolved_ideal": min(len(peaks(ideal_probs(phases, S), S, PEAK_THR)), d) / d}
                for t in THR_CHECK:
                    row[f"y_resolved_thr{int(t*100):02d}"] = min(len(peaks(P, S, t)), d) / d
                rows.append(row)
        print(f"seed {seed}, a = {a} ({S} slices): {len(gates)} gates in {time.time()-t0:.0f} s")

dataset = pd.DataFrame(rows)
path = os.path.join(SAVE_DIR, "dataset.csv")
dataset.to_csv(path, index=False)
print(f"\nSaved {dataset.shape[0]} rows x {dataset.shape[1]} columns to {path}  "
      f"(total {time.time()-t_all:.0f} s)")

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)
print("\nFirst 10 rows:")
print(dataset.head(10).to_string(index=False))

Mounted at /content/drive
seed 42, a = 3 (8 slices): 1756 gates in 8 s
seed 42, a = 4 (16 slices): 1756 gates in 8 s
seed 42, a = 5 (32 slices): 1756 gates in 13 s
seed 43, a = 3 (8 slices): 1756 gates in 14 s
seed 43, a = 4 (16 slices): 1756 gates in 15 s
seed 43, a = 5 (32 slices): 1756 gates in 11 s
seed 44, a = 3 (8 slices): 1756 gates in 8 s
seed 44, a = 4 (16 slices): 1756 gates in 10 s
seed 44, a = 5 (32 slices): 1756 gates in 12 s

Saved 15804 rows x 20 columns to /content/drive/MyDrive/Colab Notebooks/dataset.csv  (total 103 s)

First 10 rows:
 seed  a  slices     gate    class    SCM    gap  phase_std  grid_mean  grid_max     g1     g2     g3     g4  y_peaks  y_resolved  y_phase_error  y_resolved_ideal  y_resolved_thr03  y_resolved_thr08
   42  3       8     CNOT standard 0.6667 0.0000     1.0000     0.0000    0.0000 0.0000 0.0000 3.1416 3.1416        2      0.5000         0.0000            0.5000            0.5000            0.5000
   42  3       8     SWAP standard 0.6667 0

In [ ]:
# =====================================================================
import os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                              GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, balanced_accuracy_score
warnings.filterwarnings("ignore")

# ---------------- Load ----------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks"   # same folder as Cell 1
except ImportError:
    SAVE_DIR = os.getcwd()
df = pd.read_csv(os.path.join(SAVE_DIR, "dataset.csv"))
df = df[df["class"] != "standard"]                         # standard gates = reference only
print(f"Loaded {len(df)} rows (seeds {sorted(df.seed.unique())}, a = {sorted(df.a.unique())})")

X_COLS = ["SCM", "gap", "phase_std"]                       # paper features only
GAP_COLS = ["g1", "g2", "g3", "g4"]                        # used only by the physics baseline
d = 4

# ---------------- Physics baseline (no ML) ----------------
# "Two neighbouring eigenvalues form separate peaks if their gap >= c * slice width."
# number of peaks = number of big gaps (at least 1). c is tuned on the training folds only.
C_GRID = np.round(np.arange(0.2, 3.01, 0.05), 2)

def baseline_peaks(gaps, S, c):
    n = (gaps >= c * 2*np.pi / S).sum(axis=1)
    return np.clip(np.where(n == 0, 1, n), 1, d)

def fit_c(gaps, S, y_peaks):
    acc = [np.mean(baseline_peaks(gaps, S, c) == y_peaks) for c in C_GRID]
    return C_GRID[int(np.argmax(acc))]

# ---------------- Models ----------------
REG = {"Linear": lambda: LinearRegression(),
       "RandomForest": lambda: RandomForestRegressor(n_estimators=200, max_depth=5, random_state=0),
       "GradBoost": lambda: GradientBoostingRegressor(random_state=0)}
CLF = {"Majority": None,
       "Logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
       "RandomForest": lambda: RandomForestClassifier(n_estimators=200, max_depth=5, random_state=0),
       "GradBoost": lambda: GradientBoostingClassifier(random_state=0)}

rows = []
for (seed, a), dsa in df.groupby(["seed", "a"]):
    S = 2**a
    for subset, sub in [("haar_only", dsa[dsa["class"] == "haar"]), ("all_random", dsa)]:
        X, G = sub[X_COLS].to_numpy(), sub[GAP_COLS].to_numpy()
        y_res, y_err, y_pk = (sub["y_resolved"].to_numpy(), sub["y_phase_error"].to_numpy(),
                              sub["y_peaks"].to_numpy())
        # ----- Regression (R2, MAE) -----
        for target, y in [("y_resolved", y_res), ("y_phase_error", y_err)]:
            models = list(REG) + (["Baseline"] if target == "y_resolved" else [])
            pred = {m: np.zeros(len(y)) for m in models}
            for tr, te in KFold(5, shuffle=True, random_state=seed).split(X):
                for m in REG:
                    pred[m][te] = REG[m]().fit(X[tr], y[tr]).predict(X[te])
                if "Baseline" in models:
                    c = fit_c(G[tr], S, y_pk[tr])
                    pred["Baseline"][te] = baseline_peaks(G[te], S, c) / d
            for m in models:
                rows.append({"seed": seed, "a": a, "subset": subset, "task": "regression",
                             "target": target, "model": m,
                             "R2": r2_score(y, pred[m]), "MAE": mean_absolute_error(y, pred[m]),
                             "accuracy": np.nan, "balanced_accuracy": np.nan})
        # ----- Classification (number of peaks 1..4) -----
        pred = {m: np.zeros(len(y_pk), dtype=int) for m in list(CLF) + ["Baseline"]}
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y_pk):
            vals, cnts = np.unique(y_pk[tr], return_counts=True)
            pred["Majority"][te] = vals[np.argmax(cnts)]
            for m in ["Logistic", "RandomForest", "GradBoost"]:
                if len(vals) == 1:
                    pred[m][te] = vals[0]
                else:
                    pred[m][te] = CLF[m]().fit(X[tr], y_pk[tr]).predict(X[te])
            c = fit_c(G[tr], S, y_pk[tr])
            pred["Baseline"][te] = baseline_peaks(G[te], S, c)
        for m in pred:
            rows.append({"seed": seed, "a": a, "subset": subset, "task": "classification",
                         "target": "y_peaks", "model": m, "R2": np.nan, "MAE": np.nan,
                         "accuracy": accuracy_score(y_pk, pred[m]),
                         "balanced_accuracy": balanced_accuracy_score(y_pk, pred[m])})

results = pd.DataFrame(rows)

# Threshold-sensitivity check (a = 4, Haar only): does the main result depend on 0.05?
sens = []
for col in ["y_resolved_thr03", "y_resolved", "y_resolved_thr08"]:
    for seed in sorted(df.seed.unique()):
        h = df[(df.seed == seed) & (df.a == 4) & (df["class"] == "haar")]
        X, y, pr = h[X_COLS].to_numpy(), h[col].to_numpy(), np.zeros(len(h))
        for tr, te in KFold(5, shuffle=True, random_state=seed).split(X):
            pr[te] = REG["RandomForest"]().fit(X[tr], y[tr]).predict(X[te])
        results = pd.concat([results, pd.DataFrame([{
            "seed": seed, "a": 4, "subset": "haar_only", "task": "threshold_check",
            "target": col, "model": "RandomForest", "R2": r2_score(y, pr),
            "MAE": mean_absolute_error(y, pr), "accuracy": np.nan, "balanced_accuracy": np.nan}])],
            ignore_index=True)

path = os.path.join(SAVE_DIR, "results.csv")
results.to_csv(path, index=False)
print(f"Saved {len(results)} result rows to {path}")

# ---------------- Print summary (mean ± std over seeds) ----------------
pd.set_option("display.width", 220); pd.set_option("display.float_format", "{:.3f}".format)
def summary(task, metrics):
    s = (results[results.task == task].groupby(["a", "subset", "target", "model"])[metrics]
         .agg(["mean", "std"]))
    s.columns = [f"{m}_{k}" for m, k in s.columns]
    return s.reset_index()

print("\n=== Regression (mean ± std over seeds) ===")
print(summary("regression", ["R2", "MAE"]).to_string(index=False))
print("\n=== Classification: number of peaks (mean ± std over seeds) ===")
print(summary("classification", ["accuracy", "balanced_accuracy"]).to_string(index=False))
print("\n=== Peak-threshold sensitivity (RandomForest, a = 4, Haar only) ===")
print(summary("threshold_check", ["R2", "MAE"]).to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 15750 rows (seeds [np.int64(42), np.int64(43), np.int64(44)], a = [np.int64(3), np.int64(4), np.int64(5)])
Saved 225 result rows to /content/drive/MyDrive/Colab Notebooks/results.csv

=== Regression (mean ± std over seeds) ===
 a     subset        target        model  R2_mean  R2_std  MAE_mean  MAE_std
 3 all_random y_phase_error    GradBoost    0.366   0.015     0.104    0.002
 3 all_random y_phase_error       Linear    0.125   0.004     0.124    0.002
 3 all_random y_phase_error RandomForest    0.380   0.014     0.103    0.001
 3 all_random    y_resolved     Baseline    0.460   0.008     0.070    0.001
 3 all_random    y_resolved    GradBoost    0.678   0.008     0.078    0.001
 3 all_random    y_resolved       Linear    0.547   0.011     0.101    0.001
 3 all_random    y_resolved RandomForest    0.680   0.007     0.078    0.001
 3  haar_only y_phase